# 19 — fato_sancoes (Gold)

## Purpose
Build the `fato_sancoes` fact table by combining `silver_ceis` and `silver_cnep`,
joining with `dim_fornecedor` (temporal join per ADR-003) and `dim_tempo`.

## Grain
One row per sanction record: `cnpj_normalized + fonte + sancao_id`

## Input
- `local/data/silver/silver_ceis/data.parquet`
- `local/data/silver/silver_cnep/data.parquet`
- `local/data/gold/dim_fornecedor.parquet`
- `local/data/gold/dim_tempo.parquet`

## Output
- `local/data/gold/fato_sancoes.parquet`

## Steps
1. Imports and configuration
2. Load sources as lazy views
3. Build unified sanction base (ceis + cnep)
4. Join with dim_fornecedor (temporal join + current fallback)
5. Add dim_tempo FK, compute sancao_sk, write to Parquet
6. Validation

## Architecture Notes
- **Grain:** one row per sanction record per source system.
- **valor_multa:** only available from CNEP — NULL for CEIS by design (different legal framework).
- **is_ativa:** True when data_fim_sancao IS NULL OR data_fim_sancao > CURRENT_DATE.
- **data_referencia for temporal join:** data_inicio_sancao — when the sanction began.
- **_join_type:** same pattern as fato_contratos — 'temporal', 'current_fallback', 'unmatched'.
- Idempotent — safe to re-run (overwrites output file).


## Step 1 — Imports and configuration

In [ ]:
import duckdb
from pathlib import Path
from datetime import datetime, timezone

# ── Paths ─────────────────────────────────────────────────────────────────────
try:
    PROJECT_ROOT = Path(__vsc_ipynb_file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent

SILVER_DIR  = PROJECT_ROOT / "data" / "silver"
GOLD_DIR    = PROJECT_ROOT / "data" / "gold"
OUTPUT_PATH = GOLD_DIR / "fato_sancoes.parquet"

SILVER_CEIS    = str(SILVER_DIR / "silver_ceis" / "data.parquet").replace("\\", "/")
SILVER_CNEP    = str(SILVER_DIR / "silver_cnep" / "data.parquet").replace("\\", "/")
DIM_FORNECEDOR = str(GOLD_DIR   / "dim_fornecedor.parquet").replace("\\", "/")
DIM_TEMPO      = str(GOLD_DIR   / "dim_tempo.parquet").replace("\\", "/")
OUTPUT_PATH_STR = str(OUTPUT_PATH).replace("\\", "/")

GOLD_DIR.mkdir(parents=True, exist_ok=True)
LOADED_AT = datetime.now(timezone.utc).isoformat()

print(f"Silver ceis    : {SILVER_CEIS}")
print(f"Silver cnep    : {SILVER_CNEP}")
print(f"Dim fornecedor : {DIM_FORNECEDOR}")
print(f"Dim tempo      : {DIM_TEMPO}")
print(f"Output         : {OUTPUT_PATH}")
print(f"loaded_at      : {LOADED_AT}")

## Step 2 — Load sources as lazy views

In [ ]:
con = duckdb.connect()
con.execute("SET threads TO 4")
con.execute("SET memory_limit = '8GB'")
con.execute("SET preserve_insertion_order = false")

con.execute(f"""CREATE OR REPLACE VIEW v_ceis AS SELECT * FROM read_parquet('{SILVER_CEIS}')""")
con.execute(f"""CREATE OR REPLACE VIEW v_cnep AS SELECT * FROM read_parquet('{SILVER_CNEP}')""")
con.execute(f"""CREATE OR REPLACE VIEW v_dim_fornecedor AS SELECT * FROM read_parquet('{DIM_FORNECEDOR}')""")
con.execute(f"""CREATE OR REPLACE VIEW v_dim_tempo AS SELECT * FROM read_parquet('{DIM_TEMPO}')""")

counts = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM v_ceis)           AS ceis,
        (SELECT COUNT(*) FROM v_cnep)           AS cnep,
        (SELECT COUNT(*) FROM v_dim_fornecedor) AS dim_fornecedor,
        (SELECT COUNT(*) FROM v_dim_tempo)      AS dim_tempo
""").df()
print(counts.to_string(index=False))

## Step 3 — Build unified sanction base (ceis + cnep)

Key differences between sources:

| Field | CEIS | CNEP |
|---|---|---|
| valor_multa | not available | available |
| legal framework | administrative sanctions | financial penalties |
| grain | one sanction event | one penalty event |

Both share the same schema structure — CNEP adds `valor_multa`.


In [ ]:
print("Building unified sanction base...")
t0 = datetime.now()

con.execute("""
    CREATE OR REPLACE TABLE t_sancoes_base AS

    -- ── CEIS ─────────────────────────────────────────────────────────────────
    SELECT
        cnpj_normalized,
        sancao_id,
        tipo_sancao,
        tipo_sancao_detalhe,
        orgao_sancionador_nome,
        orgao_sancionador_uf,
        orgao_sancionador_esfera,
        orgao_sancionador_poder,
        numero_processo,
        data_inicio_sancao,
        data_fim_sancao,
        data_publicacao_sancao,
        data_transitado_julgado,
        -- is_ativa: no end date or end date in the future
        CASE
            WHEN data_fim_sancao IS NULL          THEN TRUE
            WHEN data_fim_sancao > CURRENT_DATE   THEN TRUE
            ELSE FALSE
        END                                        AS is_ativa,
        NULL::DECIMAL(15,2)                        AS valor_multa,  -- not in CEIS
        _fonte                                     AS fonte,
        _silver_loaded_at
    FROM v_ceis
    WHERE cnpj_normalized IS NOT NULL
      AND data_inicio_sancao IS NOT NULL

    UNION ALL

    -- ── CNEP ─────────────────────────────────────────────────────────────────
    SELECT
        cnpj_normalized,
        sancao_id,
        tipo_sancao,
        tipo_sancao_detalhe,
        orgao_sancionador_nome,
        orgao_sancionador_uf,
        orgao_sancionador_esfera,
        orgao_sancionador_poder,
        numero_processo,
        data_inicio_sancao,
        data_fim_sancao,
        data_publicacao_sancao,
        data_transitado_julgado,
        CASE
            WHEN data_fim_sancao IS NULL          THEN TRUE
            WHEN data_fim_sancao > CURRENT_DATE   THEN TRUE
            ELSE FALSE
        END                                        AS is_ativa,
        valor_multa,                               -- available in CNEP
        _fonte                                     AS fonte,
        _silver_loaded_at
    FROM v_cnep
    WHERE cnpj_normalized IS NOT NULL
      AND data_inicio_sancao IS NOT NULL
""")

n       = con.execute("SELECT COUNT(*) FROM t_sancoes_base").fetchone()[0]
elapsed = (datetime.now() - t0).total_seconds()

dist = con.execute("""
    SELECT fonte, is_ativa, COUNT(*) AS total
    FROM t_sancoes_base
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()

print(f"t_sancoes_base: {n:,} rows in {elapsed:.1f}s")
print()
print(dist.to_string(index=False))

## Step 4 — Join with dim_fornecedor (temporal join + current fallback)

Same join strategy as fato_contratos (ADR-003).
Reference date: `data_inicio_sancao` — when the sanction began.


In [ ]:
print("Joining with dim_fornecedor...")
t0 = datetime.now()

con.execute("""
    CREATE OR REPLACE TABLE t_sancoes_with_supplier AS
    WITH temporal AS (
        SELECT
            s.*,
            d.supplier_sk   AS supplier_sk,
            'temporal'      AS _join_type
        FROM t_sancoes_base s
        INNER JOIN v_dim_fornecedor d
            ON  d.cnpj_normalized = s.cnpj_normalized
            AND d.valid_from     <= s.data_inicio_sancao
            AND d.valid_to       >  s.data_inicio_sancao
    ),
    fallback AS (
        SELECT
            s.*,
            d.supplier_sk        AS supplier_sk,
            'current_fallback'   AS _join_type
        FROM t_sancoes_base s
        LEFT JOIN temporal t
            ON t.cnpj_normalized   = s.cnpj_normalized
            AND t.sancao_id        = s.sancao_id
            AND t.fonte            = s.fonte
        INNER JOIN v_dim_fornecedor d
            ON  d.cnpj_normalized = s.cnpj_normalized
            AND d.is_current      = TRUE
        WHERE t.cnpj_normalized IS NULL
    ),
    unmatched AS (
        SELECT
            s.*,
            NULL::VARCHAR        AS supplier_sk,
            'unmatched'          AS _join_type
        FROM t_sancoes_base s
        LEFT JOIN temporal  t ON t.cnpj_normalized = s.cnpj_normalized AND t.sancao_id = s.sancao_id AND t.fonte = s.fonte
        LEFT JOIN fallback  f ON f.cnpj_normalized = s.cnpj_normalized AND f.sancao_id = s.sancao_id AND f.fonte = s.fonte
        WHERE t.cnpj_normalized IS NULL
          AND f.cnpj_normalized IS NULL
    )
    SELECT * FROM temporal
    UNION ALL
    SELECT * FROM fallback
    UNION ALL
    SELECT * FROM unmatched
""")

n       = con.execute("SELECT COUNT(*) FROM t_sancoes_with_supplier").fetchone()[0]
elapsed = (datetime.now() - t0).total_seconds()

join_dist = con.execute("""
    SELECT _join_type, COUNT(*) AS total
    FROM t_sancoes_with_supplier
    GROUP BY 1 ORDER BY 2 DESC
""").df()

print(f"t_sancoes_with_supplier: {n:,} rows in {elapsed:.1f}s")
print()
print(join_dist.to_string(index=False))

## Step 5 — Add dim_tempo FK, compute sancao_sk, write to Parquet

In [ ]:
print("Writing final fato_sancoes...")
t0 = datetime.now()

con.execute(f"""
    COPY (
        SELECT
            -- Surrogate key
            md5(
                COALESCE(cnpj_normalized, '') || '|' ||
                COALESCE(sancao_id,       '') || '|' ||
                COALESCE(fonte,           '')
            )::VARCHAR                                      AS sancao_sk,

            -- Dimension FKs
            supplier_sk,
            CAST(strftime(data_inicio_sancao, '%Y%m%d') AS INTEGER) AS date_id,

            -- Natural key
            cnpj_normalized,
            sancao_id,
            fonte,
            _join_type,

            -- Sanction attributes
            tipo_sancao,
            tipo_sancao_detalhe,
            orgao_sancionador_nome,
            orgao_sancionador_uf,
            orgao_sancionador_esfera,
            orgao_sancionador_poder,
            numero_processo,

            -- Dates
            data_inicio_sancao,
            data_fim_sancao,
            data_publicacao_sancao,
            data_transitado_julgado,

            -- Status flag
            is_ativa,

            -- Measure (CNEP only — NULL for CEIS)
            valor_multa,

            -- Audit
            '{LOADED_AT}'::TIMESTAMPTZ                      AS _loaded_at

        FROM t_sancoes_with_supplier
    ) TO '{OUTPUT_PATH_STR}' (FORMAT PARQUET)
""")

con.execute("DROP TABLE IF EXISTS t_sancoes_base")
con.execute("DROP TABLE IF EXISTS t_sancoes_with_supplier")

file_size_mb = OUTPUT_PATH.stat().st_size / 1_048_576
elapsed      = (datetime.now() - t0).total_seconds()
print(f"Written : {OUTPUT_PATH}")
print(f"Size    : {file_size_mb:.2f} MB")
print(f"Time    : {elapsed:.1f}s")

## Step 6 — Validation

In [ ]:
checks = []

total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')").fetchone()[0]

n_ceis = con.execute("SELECT COUNT(*) FROM v_ceis WHERE cnpj_normalized IS NOT NULL AND data_inicio_sancao IS NOT NULL").fetchone()[0]
n_cnep = con.execute("SELECT COUNT(*) FROM v_cnep WHERE cnpj_normalized IS NOT NULL AND data_inicio_sancao IS NOT NULL").fetchone()[0]
expected = n_ceis + n_cnep

# CHECK 1 — row count
checks.append(("Row count = ceis + cnep", total == expected,
               f"{total:,} (expected {expected:,} = {n_ceis:,} + {n_cnep:,})"))

# CHECK 2 — sancao_sk unique
unique_sk = con.execute(f"SELECT COUNT(DISTINCT sancao_sk) FROM read_parquet('{OUTPUT_PATH_STR}')").fetchone()[0]
checks.append(("sancao_sk unique", unique_sk == total, f"{unique_sk:,} unique / {total:,} total"))

# CHECK 3 — fonte has exactly 2 values
fontes = sorted(con.execute(f"SELECT DISTINCT fonte FROM read_parquet('{OUTPUT_PATH_STR}')").df()["fonte"].tolist())
checks.append(("fonte has 2 values (ceis + cnep)", fontes == ["ceis", "cnep"], str(fontes)))

# CHECK 4 — valor_multa null only for CEIS
wrong_multa = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE fonte = 'ceis' AND valor_multa IS NOT NULL
""").fetchone()[0]
checks.append(("valor_multa null for all CEIS rows", wrong_multa == 0, f"{wrong_multa} unexpected non-null"))

# CHECK 5 — unmatched rate < 5%
unmatched = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}') WHERE _join_type = 'unmatched'
""").fetchone()[0]
unmatched_pct = unmatched / total * 100 if total > 0 else 0
checks.append(("Unmatched supplier_sk < 5%", unmatched_pct < 5.0,
               f"{unmatched:,} ({unmatched_pct:.2f}%)"))

# CHECK 6 — is_ativa not null
null_ativa = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}') WHERE is_ativa IS NULL").fetchone()[0]
checks.append(("is_ativa not null", null_ativa == 0, f"{null_ativa} nulls"))

# CHECK 7 — date_id range
out_of_range = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE date_id < 20010101 OR date_id > 20301231
""").fetchone()[0]
checks.append(("date_id in plausible range", out_of_range == 0, f"{out_of_range} out-of-range"))

# Report
print(f"\n{'CHECK':<55} {'STATUS':<8} DETAIL")
print("-" * 90)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"{name:<55} [{status}]   {detail}")
    if not passed:
        all_pass = False

print()
if all_pass:
    print("All checks PASSED — fato_sancoes ready.")
else:
    print("One or more checks FAILED — review output above.")

# Join type distribution
print()
print("Join type distribution:")
jd = con.execute(f"""
    SELECT _join_type, COUNT(*) AS total,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{OUTPUT_PATH_STR}')
    GROUP BY 1 ORDER BY 2 DESC
""").df()
print(jd.to_string(index=False))

# is_ativa distribution
print()
print("is_ativa distribution:")
ia = con.execute(f"""
    SELECT fonte, is_ativa, COUNT(*) AS total
    FROM read_parquet('{OUTPUT_PATH_STR}')
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()
print(ia.to_string(index=False))

# valor_multa stats (CNEP only)
print()
print("valor_multa stats (CNEP only):")
vm = con.execute(f"""
    SELECT
        COUNT(*)                    AS cnep_total,
        COUNT(valor_multa)          AS com_multa,
        SUM(valor_multa)            AS total_multas,
        MAX(valor_multa)            AS maior_multa
    FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE fonte = 'cnep'
""").df()
print(vm.to_string(index=False))

# Sample
print()
print("Sample rows (is_ativa=True):")
s = con.execute(f"""
    SELECT sancao_sk, cnpj_normalized, fonte, tipo_sancao,
           orgao_sancionador_nome, data_inicio_sancao, is_ativa, valor_multa, _join_type
    FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE is_ativa = true
    LIMIT 4
""").df()
print(s.to_string(index=False))